# Kepler Model for orbit Estimation

In [ ]:
import pandas as pd
import numpy as np
import json

data = pd.read_json(
    r".\orbital_elements.json"
)

with open(r".\data_manifest.json", encoding="utf-8") as f:
    metadata = json.load(f)

STEP_SECONDS = 5
DURATION_HOURS = 24
mu = 398600.4418 
EARTH_RADIUS_KM = 6378.137

Parameters of each satellite

In [ ]:
data.head()

In [ ]:
data["EPOCH"] = pd.to_datetime(data["EPOCH"], utc=True)  # desired time
data["OBJECT_NAME"].value_counts().loc[lambda counts: counts > 1]

In [ ]:
data.dtypes

Metadata of each Satellite

In [ ]:
metadata.keys()

In [ ]:
for row in data.itertuples(index=False):
    print(f"Satellite: {row.OBJECT_NAME}, Epoch: {row.EPOCH},Retrieve date: {metadata['retrieved_utc']}, Norad:{row.NORAD_CAT_ID} \n")

## Stats of Satellites

In [ ]:
data.describe()

## Time window

In [ ]:
data.head()

In [ ]:
t0 = pd.to_datetime(data["EPOCH"].max(), utc=True).floor("D")
start = t0
end = start + pd.Timedelta(hours=DURATION_HOURS)
times = pd.date_range(start=start, end=end, freq=pd.Timedelta(seconds=STEP_SECONDS))
print(f"Number of time steps: {len(times)}, Start time: {t0}, End time: {end}")
for row in data.itertuples(index=False):
    offset =  t0 - row.EPOCH
    if offset.days > 7:
        print(f"{row.OBJECT_NAME} has an offset of {offset.days} days")

## Propagation of orbits

In [ ]:
def gmst_rad(t):
    t = pd.Timestamp(t)
    if t.tzinfo is None:
        t = t.tz_localize("UTC")  # assumes a timezone-naive input is UTC
    else:
        t = t.tz_convert("UTC")

    d = t.to_julian_date() - 2451545.0  # days since J2000
    T = d / 36525.0

    theta_deg = (
        280.46061837
        + 360.98564736629 * d
        + 0.000387933 * T**2
        - T**3 / 38710000
    ) % 360

    return np.deg2rad(theta_deg)

In [ ]:
j200= pd.Timestamp("2000-01-01 12:00:00", tz="UTC")
theta_2000 = gmst_rad(j200)
print(f"GMST at J2000: {theta_2000*180/np.pi:.2f} degrees")

In [ ]:
theta = []
for t in times:
    th = gmst_rad(t)
    theta.append(th)

theta = np.array(theta)
positions = {}
for row in data.itertuples(index=False):
    x_ecef = []
    y_ecef = []
    z_ecef = []
    dt = (times - row.EPOCH).total_seconds().to_numpy()

    n = row.MEAN_MOTION * 2 * np.pi / 86400
    e = row.ECCENTRICITY
    i, Omega, omega, M0 = np.deg2rad([
        row.INCLINATION,
        row.RA_OF_ASC_NODE,
        row.ARG_OF_PERICENTER,
        row.MEAN_ANOMALY,
    ])
    
    M = (M0 + n * dt) % (2 * np.pi)
    a = (mu / n**2) ** (1 / 3)
    # Newton's method to solve E - e*sin(E) = M
    E = np.full_like(M, np.pi) if e > 0.8 else M.copy()
    max_iter = 50
    tol=1e-12
    for _ in range(max_iter):
        residual = E - e * np.sin(E) - M
        converged = np.abs(residual) < tol

        if np.all(converged):
             break

        slope = 1 - e * np.cos(E)
        change = np.where(converged, 0.0, residual / slope)
        E -= change
        residual = E - e * np.sin(E) - M
    if not np.all(np.abs(residual) < tol):
        raise RuntimeError(f"Kepler solver did not converge in {max_iter} iterations")
    x_orbit = a * (np.cos(E) - e)
    y_orbit = a * np.sqrt(1 - e**2) * np.sin(E)
    x = (
        (np.cos(Omega) * np.cos(omega) - np.sin(Omega) * np.sin(omega) * np.cos(i)) * x_orbit
        + (-np.cos(Omega) * np.sin(omega) - np.sin(Omega) * np.cos(omega) * np.cos(i)) * y_orbit
    )
    y = (
        (np.sin(Omega) * np.cos(omega) + np.cos(Omega) * np.sin(omega) * np.cos(i)) * x_orbit
        + (-np.sin(Omega) * np.sin(omega) + np.cos(Omega) * np.cos(omega) * np.cos(i)) * y_orbit
    )
    z = np.sin(omega) * np.sin(i) * x_orbit + np.cos(omega) * np.sin(i) * y_orbit


    x_f =  np.cos(theta) * x + np.sin(theta) * y
    y_f = -np.sin(theta) * x + np.cos(theta) * y
    z_f = z
    x_ecef.append(x_f)
    y_ecef.append(y_f)
    z_ecef.append(z_f)
    x_ecef = np.array(x_ecef).squeeze()
    y_ecef = np.array(y_ecef).squeeze()
    z_ecef = np.array(z_ecef).squeeze()

    r = np.sqrt(x_ecef**2 + y_ecef**2 + z_ecef**2)

    # Spherical (geocentric) latitude and longitude, degrees
    lat = np.rad2deg(np.arctan2(z_ecef, np.hypot(x_ecef, y_ecef)))
    lon = np.rad2deg(np.arctan2(y_ecef, x_ecef))
    # Approximate altitude above a spherical Earth, km
    alt = r - EARTH_RADIUS_KM

    # Two-body speed from vis-viva, km/s
    speed = np.sqrt(mu * (2 / r - 1 / a))

    ones = np.ones_like(dt)
    period_h = 2*np.pi/n/3600
    period_h = ones * period_h
    perigee_km = a*(-e +1) - EARTH_RADIUS_KM
    apogee_km = a*(1 + e) - EARTH_RADIUS_KM
    eccentricity = ones * e
    inclination = ones * np.rad2deg(i)
    norad_id = ones * row.NORAD_CAT_ID
   
    satellite_positions = pd.DataFrame({
        "time": dt,
        "timestamp": times,
        "x": x,
        "y": y,
        "z": z,
        "x_ecef": x_ecef,
        "y_ecef": y_ecef,
        "z_ecef": z_ecef,
        "lat": lat,
        "lon": lon,
        "alt": alt,
        "theta" : theta*180/np.pi,
        "speed": speed,
        "period_h": period_h,
        "perigee_km": perigee_km,
        "apogee_km": apogee_km,
        "eccentricity": eccentricity,
        "inclination": inclination,
        "norad_id": norad_id,
        "distance_from_earth_center_km": r
    })
    positions[row.OBJECT_NAME] = satellite_positions
    

At SES-5

In [ ]:
import matplotlib.pyplot as plt
SES_5 = positions["SES-5"]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(SES_5["time"], SES_5["lon"])

ax.set(
    title="SES-5 Longitude over time",
    xlabel="Time (UTC)",
    ylabel="Longitude (degrees)",
)
ax.grid(True)
fig.autofmt_xdate()
plt.show()

## Summary table

In [ ]:
cols = [
    "norad_id", "eccentricity", "perigee_km",
    "apogee_km", "period_h", "inclination",
]
summary = {}
for sat_name in positions.keys():
    df = positions[sat_name][cols].head().copy()
    #df.index = pd.Index([sat_name] * len(df), name="sat_name")
    values = df.iloc[0]
    summary[sat_name] = values
summary_df = pd.DataFrame.from_dict(summary, orient="index")
summary_df.index.name = "sat_name"
summary_df

## Ground Track and speed

In [ ]:
from pathlib import Path
import re
import pandas as pd
from io import StringIO

text = Path("world_map.txt").read_text(encoding="utf-8")
COLORS = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']
blocks = re.split(r"\n\s*\n", text.strip())
coast_segments = [
    pd.read_csv(
        StringIO(block),
        sep=r"\s+",
        header=None,
        names=["lat", "lon"],
    )
    for block in blocks
    if block.strip()
]

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for index, (sat, df) in enumerate(positions.items()):
    if sat == "SES-5":
        ax.scatter(df["lon"], df["lat"], label=sat, color=COLORS[index % len(COLORS)], s=80)
    else:
        ax.scatter(df["lon"], df["lat"], label=sat, color=COLORS[index % len(COLORS)], s=2)
for segment in coast_segments:
    ax.plot(segment["lon"], segment["lat"], color="black", linewidth=0.5)
ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)
ax.set_xlabel("Longitude (degrees)")
ax.set_ylabel("Latitude (degrees)")
ax.set_title("Multi-Satellite Ground Tracks")
ax.grid(True)
ax.legend()
fig.tight_layout()
fig.savefig("ground_tracks.png", dpi=200)
plt.show()


In [ ]:
fig, axes = plt.subplots(
     len(positions),1,
    figsize=( 10, len(positions)*5),
    squeeze=False,
)
i=0
for ax, (sat, df) in zip(axes.flat, positions.items()):
    ax.plot(df["timestamp"], df["speed"],color=COLORS[i])
    ax.set_title(sat)
    ax.set_xlabel("Time")
    ax.set_ylabel("Speed (km/s)")
    ax.grid(True)
    i=i+1
    
plt.savefig("velocity_timeseries.png", dpi=200)
plt.show()

Satellite with greatest distance to center of the Earth

In [ ]:
for key in positions.keys():
    df = positions[key]
    print(f"Satellite {key}: Maximum distance from Earth center = {df.distance_from_earth_center_km.max()} km")

## Exporting outputs

In [ ]:
from pathlib import Path
import zipfile
import xml.etree.ElementTree as ET
from matplotlib.colors import to_hex

required_names = ["data", "metadata", "positions", "summary_df", "times", "start", "end", "STEP_SECONDS", "COLORS"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise RuntimeError(f"Run the earlier notebook stages first; missing: {', '.join(missing_names)}")
if "provenance_note" not in metadata:
    raise RuntimeError("The data manifest is missing provenance_note")
if not COLORS:
    raise RuntimeError("COLORS is empty")
if STEP_SECONDS <= 0 or 60 % STEP_SECONDS != 0:
    raise ValueError("STEP_SECONDS must be a positive divisor of 60 for 60-second animation samples")

OUT_DIR = Path("satellite_orbit_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
summaries = summary_df
elements = data
source_note = metadata["provenance_note"]

KML_NS = "http://www.opengis.net/kml/2.2"
GX_NS = "http://www.google.com/kml/ext/2.2"
ET.register_namespace("", KML_NS)
ET.register_namespace("gx", GX_NS)
qk = lambda name: f"{{{KML_NS}}}{name}"
qgx = lambda name: f"{{{GX_NS}}}{name}"


def kml_color(color):
    rgb = to_hex(color, keep_alpha=False).lstrip("#")
    return f"ff{rgb[4:6]}{rgb[2:4]}{rgb[0:2]}"


def add_document(title):
    root = ET.Element(qk("kml"))
    document = ET.SubElement(root, qk("Document"))
    ET.SubElement(document, qk("name")).text = title
    return root, document


def add_style(document, style_id, color):
    style = ET.SubElement(document, qk("Style"), {"id": style_id})
    icon_style = ET.SubElement(style, qk("IconStyle"))
    ET.SubElement(icon_style, qk("color")).text = color
    ET.SubElement(icon_style, qk("scale")).text = "0.7"
    icon = ET.SubElement(icon_style, qk("Icon"))
    ET.SubElement(icon, qk("href")).text = "http://maps.google.com/mapfiles/kml/shapes/placemark_circle.png"
    line_style = ET.SubElement(style, qk("LineStyle"))
    ET.SubElement(line_style, qk("color")).text = color
    ET.SubElement(line_style, qk("width")).text = "2"


def write_kml(root, output_path):
    ET.indent(root, space="  ")
    ET.ElementTree(root).write(output_path, encoding="UTF-8", xml_declaration=True)


def summary_table(table):
    return table.to_string()


def elements_table(table):
    return table.to_string(index=False)


root_orbit, doc_orbit = add_document("Multi-Satellite Orbits | Earth-fixed")
root_animation, doc_animation = add_document("Multi-Satellite Ground Tracks | Animation")
satellite_items = list(positions.items())
if not satellite_items:
    raise RuntimeError("No satellite positions are available")

for index, (sat_name, frame) in enumerate(satellite_items):
    required_columns = {"timestamp", "lon", "lat", "alt"}
    missing_columns = required_columns.difference(frame.columns)
    if missing_columns:
        raise RuntimeError(f"{sat_name} positions are missing columns: {', '.join(sorted(missing_columns))}")
    if frame.empty:
        raise RuntimeError(f"No position samples are available for {sat_name}")

    color = kml_color(COLORS[index % len(COLORS)])
    style_id = f"satellite_{index}"
    add_style(doc_orbit, style_id, color)
    add_style(doc_animation, style_id, color)

    placemark = ET.SubElement(doc_orbit, qk("Placemark"))
    ET.SubElement(placemark, qk("name")).text = sat_name
    ET.SubElement(placemark, qk("styleUrl")).text = f"#{style_id}"
    line = ET.SubElement(placemark, qk("LineString"))
    ET.SubElement(line, qk("tessellate")).text = "1"
    ET.SubElement(line, qk("altitudeMode")).text = "absolute"
    coordinates = []
    for lon, lat, alt_km in zip(frame["lon"], frame["lat"], frame["alt"]):
        if not np.all(np.isfinite([lon, lat, alt_km])):
            raise ValueError(f"Non-finite orbit coordinate found for {sat_name}")
        coordinates.append(f"{lon:.8f},{lat:.8f},{alt_km * 1000:.3f}")
    ET.SubElement(line, qk("coordinates")).text = " ".join(coordinates)

    sampled = frame.iloc[::60 // STEP_SECONDS].reset_index(drop=True)
    animation_placemark = ET.SubElement(doc_animation, qk("Placemark"))
    ET.SubElement(animation_placemark, qk("name")).text = sat_name
    ET.SubElement(animation_placemark, qk("styleUrl")).text = f"#{style_id}"
    multi_track = ET.SubElement(animation_placemark, qgx("MultiTrack"))
    jumps = np.flatnonzero(np.abs(np.diff(sampled["lon"].to_numpy(dtype=float))) > 180) + 1
    for segment_indices in np.split(np.arange(len(sampled)), jumps):
        if len(segment_indices) == 0:
            continue
        track = ET.SubElement(multi_track, qgx("Track"))
        ET.SubElement(track, qk("altitudeMode")).text = "clampToGround"
        for point_index in segment_indices:
            point = sampled.iloc[point_index]
            timestamp = pd.Timestamp(point["timestamp"])
            if timestamp.tzinfo is None:
                raise ValueError(f"Timestamp for {sat_name} is missing timezone information")
            timestamp = timestamp.tz_convert("UTC")
            ET.SubElement(track, qk("when")).text = timestamp.strftime("%Y-%m-%dT%H:%M:%SZ")
            ET.SubElement(track, qgx("coord")).text = f"{point['lon']:.8f} {point['lat']:.8f} 0"

orbit_kml = OUT_DIR / "orbit_tracks_ecef.kml"
animation_kml = OUT_DIR / "orbit_animation.kml"
write_kml(root_orbit, orbit_kml)
write_kml(root_animation, animation_kml)

report_path = OUT_DIR / "orbit_report.txt"
report_text = "\n".join([
    "Satellite Orbit Report - built during Space Career Week 2026, GNSS Academy",
    "",
    f"Source: {source_note}",
    f"Window: {start.isoformat()} to {end.isoformat()}",
    f"Step: {STEP_SECONDS} seconds",
    "Model caveat: Two-body Kepler propagation and spherical-Earth coordinates are approximate and do not replace SGP4.",
    "",
    "Summary table",
    summary_table(summaries),
    "",
    "Orbital elements",
    elements_table(elements),
    "",
])
report_path.write_text(report_text, encoding="utf-8")

output_paths = [orbit_kml, animation_kml, report_path]
zip_path = OUT_DIR / "satellite_orbit_outputs.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for output_path in output_paths:
        archive.write(output_path, arcname=output_path.name)

for output_path in [*output_paths, zip_path]:
    if not output_path.is_file() or output_path.stat().st_size == 0:
        raise RuntimeError(f"Output was not created correctly: {output_path}")
ET.parse(orbit_kml)
ET.parse(animation_kml)
print(f"Created {len(output_paths)} files and {zip_path}")
